1. Load a Convolutional Neural Network

In [1]:
import pandas as pd

# Read all classes from csv file
dataframe = pd.read_csv('./Multi_Label_dataset/train_data.csv')
classes = dataframe.columns[2:]

In [2]:
from torchvision.models import resnet50
import torch.nn as nn
import torch

# New weights with accuracy 80.858%
model = resnet50()
model.fc = nn.Linear(2048, len(classes))
model.load_state_dict(torch.load('./multi_classification_model.pt'))


<All keys matched successfully>

In [3]:
import torchvision.transforms.v2 as transforms
import torch

transform = transforms.Compose([
    transforms.Resize((425, 300)),
    transforms.ToImage(),  # Convert to image format
    transforms.ConvertImageDtype(torch.float32),  # Convert image dtype
    transforms.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5))])

In [4]:
from PIL import Image
from torch.nn.functional import sigmoid

img_path = 'Multi_Label_dataset/tt0085255.jpg'
image = Image.open(img_path).convert('RGB')
image = transform(image)

# Check if GPU (cuda) is available
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

# Move our model to device (GPU or CPU)
model.to(device)
model.eval()

image = image.to(device)
image = image.unsqueeze(0)

output = model(image)

# Normalize range
output = sigmoid(output)

# get predict class based on threshold
output = output.squeeze(0)
output = torch.nonzero(output > 0.4, as_tuple=True)[0].cpu()

if output.shape[0] > 1:
    genre_list = classes[output].to_list()
    print("The genres of this poster are", ', '.join(genre_list))
elif output.shape[0] == 1:
    print("The genre of this poster is", classes[output.item()])
else:
    print("This poster does not match any genres")

This poster does not match any genres
